# Practice 39 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — spot the bug

Load `sns.load_dataset('penguins')`. Drop nulls. Each snippet has one bug — fix it and explain in a comment.

```python
# A — fraction of penguins above median body_mass_g
(penguins['body_mass_g'] > penguins['body_mass_g'].mean()).mean()

# B — label penguins by bill_depth: 'deep' (>18), 'mid' (15–18), 'shallow' (<15)
conditions = [penguins['bill_depth_mm'] > 15, penguins['bill_depth_mm'] > 18]
choices = ['mid', 'deep']
penguins['depth_label'] = np.select(conditions, choices, default='shallow')

# C — species with the highest mean flipper_length_mm
penguins.groupby('species')['flipper_length_mm'].mean().idxmin()

# D — add log_mass, then correlation with flipper_length_mm
penguins['log_mass'] = np.log(penguins['body_mass_g'])
np.corrcoef(penguins['log_mass'], penguins['flipper_length_mm'])
```

In [ ]:
penguins = sns.load_dataset('penguins').dropna()

# Your fixes here
#A
(penguins['body_mass_g']>penguins['body_mass_g'].median()).mean()
# median not mean 

#B
conditions = [penguins['bill_depth_mm']>18, penguins['bill_depth_mm']>15]
choices = ['deep','mid']
penguins['depth_label'] = np.select(conditions, choices, default = 'shallow')
# evalutes the first one first and the rest, so need to be correctly ordered

#C
penguins.groupby('species')['flipper_length_mm'].mean().idxmax()
# idxmax not idxmin

#D
#looks correct



np.float64(0.48348348348348347)

---
## Level 2 — iris + mpg: open questions

**Part A — iris**

Load `sns.load_dataset('iris')`.

1. Add `area_approx` = `petal_length * petal_width` vectorized. Which species has the highest mean `area_approx`? Lowest?
2. Use `transform` to add each flower's species-level mean `sepal_length`. Add `sepal_diff` = `sepal_length - species mean`. Which species has the highest std of `sepal_diff`?
3. Use a set comprehension to find species where mean `petal_length` > 3.

**Part B — mpg**

Load `sns.load_dataset('mpg')`. Drop nulls.

4. Add `power_to_weight` = `horsepower / weight * 100` vectorized. Use `np.select()` to add `pw_class`: `'high'` (> 4), `'mid'` (2–4), `'low'` (< 2).
5. What is the mean `mpg` per `pw_class`? Does more power-to-weight mean worse fuel economy?
6. Convert `origin` and `pw_class` to `category`. Print memory before and after in KB.

In [6]:
# Your code here

iris = sns.load_dataset('iris')

iris['area_approx'] = iris['petal_length']*iris['petal_width']
iris.groupby("species")['area_approx'].mean().idxmax()
iris.groupby("species")['area_approx'].mean().idxmin()

iris['sm_sl'] = iris.groupby('species')['sepal_length'].transform('mean')
iris['sepal_diff'] = iris['sepal_length'] - iris['sm_sl']
iris.groupby('species')['sepal_diff'].std().idxmax()

mpl = iris.groupby('species')['petal_length'].mean()

{name for name, m in zip(mpl.index, mpl) if m>3}

{'versicolor', 'virginica'}

In [27]:
mpg = sns.load_dataset("mpg").dropna()
mpg['power_to_weight'] = mpg['horsepower']/mpg['weight']*100
conds = [mpg['power_to_weight']>4, mpg['power_to_weight']>=2]
chos = ['high','mid']
mpg['pw_class'] = np.select(conds, chos, default='low')

mpg.groupby('pw_class')['mpg'].mean()
b = mpg.memory_usage(deep = True).sum()/1024
print(f'before: {b:.2f}')
mpg[['origin','pw_class']] = mpg[['origin','pw_class']].astype('category')

aa = mpg.memory_usage(deep = True).sum()/1024
print(f'after: {aa:.2f}')


before: 101.91
after: 56.83


---
## Level 3 — full pipeline

Load `sns.load_dataset('titanic')`. Write a `.pipe()` chain — no `.apply()`.

- **`engineer(df)`** — add `family_size` (sibsp + parch + 1), `is_alone` (family_size == 1), `fare_bracket` with `np.select()`: `'high'` (fare > `np.percentile(fare, 75)`), `'mid'` (fare > `np.percentile(fare, 25)`), `'low'` otherwise. Convert `sex` and `fare_bracket` to `category`.
- **`analyze(df)`** — add `survival_score`: `2` if survived and not is_alone, `1` if survived and is_alone, `0` if not survived. Use `np.select()`.

After the chain:
- Pivot table: mean `survival_score` by `sex` × `fare_bracket`, `observed=True`. Who has the highest mean survival score?
- Use a set comprehension to find all `fare_bracket` values where mean `survival_score` > 1.
- Among passengers travelling alone, what is `np.corrcoef` between `fare` and `survived`?

In [28]:
# Your code here


titanic = sns.load_dataset('titanic')

def engineer(df):
    df['family_size'] = df['sibsp']+df['parch']+1
    df['is_alone'] = df['family_size']==1
    cos = [df['fare']> np.percentile(df['fare'],75), df['fare']> np.percentile(df['fare'],25)]
    chs = ['high', 'mid']
    df['fare_bracket'] = np.select(cos, chs,default = 'low')
    df[['sex','fare_bracket']] = df[['sex','fare_bracket']].astype('category')
    return df 

def analyze(df):
    conds = [(df['survived'])&(~df['is_alone']), (df['survived'])&(df['is_alone'])]
    chos = [2, 1]
    df['survival_score'] = np.select(conds, chos, default=0)
    return df 

res = titanic.copy().pipe(engineer).pipe(analyze)
p = res.pivot_table(
    index = 'sex',
    columns = 'fare_bracket',
    values = 'survival_score',
    observed = True
)
p.stack().idxmax()


ms = res.groupby('fare_bracket', observed=True)['survival_score'].mean()
{name for name, v in zip(ms.index, ms) if v >1}

alone = res.loc[res['is_alone']]
np.corrcoef(alone['fare'], alone['survived'])[0,1]

np.float64(0.2555015847557568)